# Bronze Layer — Ingesta con Auto Loader

Ingesta incremental de datos de Instagram Reels desde el volumen de origen hacia la tabla `bronze_daniel` usando **Auto Loader** con esquema explícito y `cloudFiles.schemaEvolutionMode = "none"`. Incluye la creación de los schemas bronze, silver y gold para la arquitectura multi-capa.

In [0]:
# Configuración reutilizable mediante widgets

dbutils.widgets.text("catalog_name", "dtasociate", "Catalog name")
dbutils.widgets.text("bronze_schema", "bronze", "Bronze schema")
dbutils.widgets.text("silver_schema", "silver", "Silver schema")
dbutils.widgets.text("gold_schema", "gold", "Gold schema")
dbutils.widgets.text("bronze_table", "bronze_daniel", "Bronze table name")
dbutils.widgets.text("source_path", "/Volumes/dtasociate/default/sources/", "Source path")

catalog_name   = dbutils.widgets.get("catalog_name")
bronze_schema  = dbutils.widgets.get("bronze_schema")
silver_schema  = dbutils.widgets.get("silver_schema")
gold_schema    = dbutils.widgets.get("gold_schema")
bronze_table   = dbutils.widgets.get("bronze_table")
source_path    = dbutils.widgets.get("source_path")

# Nombres fully-qualified
bronze_full_table = f"{catalog_name}.{bronze_schema}.{bronze_table}"

# Ubicación de checkpoints dentro del volumen
checkpoint_path   = f"/Volumes/{catalog_name}/default/checkpoints/{bronze_table}/"

print(f"Catalog:          {catalog_name}")
print(f"Bronze table:     {bronze_full_table}")
print(f"Source path:      {source_path}")
print(f"Checkpoint path:  {checkpoint_path}")

## Configuración de los widgets

| Parámetro          | Valor                                                    |
|--------------------|----------------------------------------------------------|
| **Catalog**        | `dtasociate`                                              |
| **Bronze table**   | `dtasociate.bronze.bronze_daniel`                         |
| **Source path**    | `/Volumes/dtasociate/default/sources/`                   |
| **Checkpoint path**| `/Volumes/dtasociate/default/checkpoints/bronze_daniel/` |

![image_1791410320554.png](./image_1791410320554.png "image_1791410320554.png")

In [0]:
# Crear catálogo y schemas (bronze, silver, gold)

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"USE CATALOG {catalog_name}")

for schema in [bronze_schema, silver_schema, gold_schema]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    print(f"Schema '{schema}' creado/verificado")

# Crear volumen para checkpoints si no existe
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.default.checkpoints")
print(f"Volumen '{catalog_name}.default.checkpoints' creado/verificado")

print(f"\nCatálogo '{catalog_name}' listo con schemas: {bronze_schema}, {silver_schema}, {gold_schema}")

## Resultados de la creación del catálogo y schemas

| Recurso                              | Estado            |
|--------------------------------------|-------------------|
| Schema `bronze`                       | Creado/verificado |
| Schema `silver`                      | Creado/verificado |
| Schema `gold`                        | Creado/verificado |
| Volumen `dtasociate.default.checkpoints` | Creado/verificado |

> **Catálogo `dtasociate`** listo con schemas: `bronze`, `silver`, `gold`

In [0]:
# ============================================
# Esquema explícito para Auto Loader
# cloudFiles.schemaEvolutionMode = "none" requiere esquema explícito
# ============================================

from pyspark.sql.types import (
    StructType, StructField,
    LongType, StringType, TimestampType,
    DoubleType, IntegerType
)

explicit_schema = StructType([
    StructField("post_id",              LongType(),      True),
    StructField("shortcode",            StringType(),    True),
    StructField("creator_handle",       StringType(),    True),
    StructField("caption",              StringType(),    True),
    StructField("timestamp_utc",       TimestampType(), True),
    StructField("video_duration_sec",  DoubleType(),    True),
    StructField("views",               IntegerType(),   True),
    StructField("likes",               IntegerType(),   True),
    StructField("comments",             IntegerType(),   True),
    StructField("caption_word_count",  IntegerType(),   True),
    StructField("hashtag_count",       IntegerType(),   True),
    StructField("niche",                StringType(),    True),
    StructField("engagement_rate_pct",  DoubleType(),    True),
    StructField("virality_score",      DoubleType(),    True),
    StructField("virality_tier",        StringType(),    True),
    StructField("_rescued_data",        StringType(),    True),
])

print("Esquema explícito definido con 16 campos")

In [0]:
# Auto Loader — Ingesta incremental a bronze

from pyspark.sql.functions import current_timestamp

stream_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaEvolutionMode", "none")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}schema/")
    .option("header", "true")
    .option("multiLine", "true")
    .option("pathGlobFilter", "*.csv")
    .schema(explicit_schema)
    .load(source_path)
    # Metadata de auditoría bronze
    .withColumn("_ingest_timestamp", current_timestamp())
)

query = (stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path) # Ensure 'checkpoint_path' is a valid existing volume or directory
    .option("mergeSchema", "false")
    .trigger(availableNow=True)  # Procesa archivos disponibles y se detiene (incremental en re-ejecuciones)
    .toTable(bronze_full_table)
)

query.awaitTermination()
print(f"Ingesta completada → {bronze_full_table}")

In [0]:
# Verificar datos ingestados en bronze

row_count = spark.sql(f"SELECT COUNT(*) AS total_rows FROM {bronze_full_table}").collect()[0]["total_rows"]
print(f"Total de filas en {bronze_full_table}: {row_count}")

display(spark.sql(f"SELECT * FROM {bronze_full_table} LIMIT 10"))

## Resultados de la verificación

**Total de filas en `dtasociate.bronze.bronze_daniel`: 88**

| post_id | shortcode | creator_handle | timestamp_utc | video_duration_sec | views | likes | comments | caption_word_count | hashtag_count | niche | engagement_rate_pct | virality_score | virality_tier | _ingest_timestamp |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 3971607901016940000 | Dcd_FfvP96F | techburner | 2026-08-25T15:44:22 | 51.4 | 3198411 | 224480 | 1073 | 2 | 4 | Technology & Gadgets | 7.05 | 88.2 | Viral Hit | 2026-10-07T21:44:18 |
| 3968684083096105000 | DcTmSWGsLCf | ranveerallahbadia | 2026-08-21T14:55:37 | 100.7 | 2969964 | 284682 | 583 | 116 | 3 | Entertainment | 9.6 | 98.1 | Viral Hit | 2026-10-07T21:44:18 |
| 3921292129996925000 | DZrOmqVAGjH | beerbiceps | 2026-06-17T05:37:39 | 50.7 | 1982315 | 371478 | 3853 | 12 | 3 | Fitness & Lifestyle | 18.93 | 98.1 | Viral Hit | 2026-10-07T21:44:18 |
| 3957069018352962000 | DbqVU6Gocm7 | onepercentclubshow | 2026-08-05T14:17:47 | 40.7 | 1606642 | 47697 | 304 | 40 | 0 | Entertainment | 2.99 | 69.2 | High Reach | 2026-10-07T21:44:18 |
| 3972244393712137700 | DcgPzrwP7YO | techburner | 2026-08-26T12:48:43 | 37.2 | 1588449 | 168964 | 350 | 4 | 1 | Technology & Gadgets | 10.66 | 97.2 | Viral Hit | 2026-10-07T21:44:18 |
| 3970151839831437000 | DcY0BBFhC1k | beerbiceps | 2026-08-23T15:30:06 | 45.2 | 1385190 | 87163 | 422 | 31 | 2 | Fitness & Lifestyle | 6.32 | 81.9 | Viral Hit | 2026-10-07T21:44:18 |
| 3320693160942413000 | C4VeNYKCqQd | kabitaskitchen | 2024-03-10T13:33:13 | 31.6 | 1378075 | 130386 | 1936 | 11 | 14 | Food & Cooking | 9.6 | 95.0 | Viral Hit | 2026-10-07T21:44:18 |
| 3969412398414500000 | DcWL4umhxBc | beerbiceps | 2026-08-22T15:00:15 | 35.0 | 1362541 | 120742 | 592 | 36 | 4 | Fitness & Lifestyle | 8.9 | 92.2 | Viral Hit | 2026-10-07T21:44:18 |
| 3911367908586589000 | DZH-GO5KosS | beerbiceps | 2026-06-03T12:58:39 | 19.6 | 1305768 | 178880 | 1070 | 79 | 1 | Fitness & Lifestyle | 13.78 | 96.4 | Viral Hit | 2026-10-07T21:44:18 |
| 3975950150500380700 | DctaZitPHAM | techburner | 2026-08-31T15:31:59 | 35.7 | 1215340 | 81946 | 184 | 4 | 1 | Technology & Gadgets | 6.76 | 83.2 | Viral Hit | 2026-10-07T21:44:18 |

> \*Columnas `caption` y `_rescued_data` omitidas por legibilidad. Mostrando 10 de 88 filas.\*
